# Task 2: Document Extraction and Knowledge Integration (Knowledge Graph/Ontology)

Prototype pipeline for extracting entities/relations from documents and mapping them onto a knowledge graph schema.

- **Dataset:** [DocRED](https://huggingface.co/datasets/thunlp/docred) — document-level entity/relation extraction, built from Wikipedia, with relations mapped to real Wikidata properties (a genuine ontology, not an ad hoc one).
- **Baseline extractor:** spaCy NER, run on raw document text (ignoring DocRED's gold labels) to simulate "we don't have labels yet."
- **Flow:** load dataset → inspect gold entities/relations for one document → run baseline extraction on the same document's raw text → compare baseline output against the gold Wikidata-based schema → log coverage gaps → findings write-up (bottom of notebook).

### Step 1: Load DocRED and inspect one document's gold annotations

In [1]:
%pip install datasets spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 111.7 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
from datasets import load_dataset

# train_annotated = human-annotated split (gold entities + relations),
# as opposed to train_distant which is noisier weak supervision.
ds = load_dataset("thunlp/docred", revision="refs/convert/parquet")
ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


DatasetDict({
    train: Dataset({
        features: ['title', 'sents', 'vertexSet', 'labels'],
        num_rows: 104926
    })
    validation: Dataset({
        features: ['title', 'sents', 'vertexSet', 'labels'],
        num_rows: 998
    })
    test: Dataset({
        features: ['title', 'sents', 'vertexSet', 'labels'],
        num_rows: 1000
    })
})

In [3]:
# The parquet conversion merges the original train_annotated (gold, human-labeled) and
# train_distant (noisier, weakly-supervised) splits into a single "train" split.
# 104,926 rows = 3,053 (train_annotated) + 101,873 (train_distant), in that order.
# Slice explicitly so we always work with gold-quality annotations, not distant supervision.
NUM_ANNOTATED = 3053
train_annotated = ds["train"].select(range(NUM_ANNOTATED))
print(f"Using {len(train_annotated)} gold-annotated documents (train_annotated portion only)")

Using 3053 gold-annotated documents (train_annotated portion only)


In [4]:
doc = train_annotated[0]

# sents is a list of tokenized sentences; join into plain text for the baseline extractor later
doc_text = " ".join(" ".join(sent) for sent in doc["sents"])

print("Title:", doc["title"])
print("\nText:\n", doc_text)

Title: AirAsia Zest

Text:
 Zest Airways , Inc. operated as AirAsia Zest ( formerly Asian Spirit and Zest Air ) , was a low - cost airline based at the Ninoy Aquino International Airport in Pasay City , Metro Manila in the Philippines . It operated scheduled domestic and international tourist services , mainly feeder services linking Manila and Cebu with 24 domestic destinations in support of the trunk route operations of other airlines . In 2013 , the airline became an affiliate of Philippines AirAsia operating their brand separately . Its main base was Ninoy Aquino International Airport , Manila . The airline was founded as Asian Spirit , the first airline in the Philippines to be run as a cooperative . On August 16 , 2013 , the Civil Aviation Authority of the Philippines ( CAAP ) , the regulating body of the Government of the Republic of the Philippines for civil aviation , suspended Zest Air flights until further notice because of safety issues . Less than a year after AirAsia and 

In [5]:
# vertexSet: one entry per gold entity, each with a "name" and NER "type" (per mention)
gold_entities = [(v[0]["name"], v[0]["type"]) for v in doc["vertexSet"]]
print(f"{len(gold_entities)} gold entities:")
for name, etype in gold_entities:
    print(f"  {name!r}  ({etype})")

17 gold entities:
  'Zest Airways, Inc.'  (ORG)
  'Ninoy Aquino International Airport'  (LOC)
  'Pasay City'  (LOC)
  'Metro Manila'  (LOC)
  'Philippines'  (LOC)
  'Manila'  (LOC)
  'Cebu'  (LOC)
  '24'  (NUM)
  '2013'  (TIME)
  'Philippines AirAsia'  (ORG)
  'Asian Spirit'  (ORG)
  'Civil Aviation Authority of the Philippines'  (ORG)
  'Zest Air'  (ORG)
  'a year'  (NUM)
  'AirAsia'  (ORG)
  'AirAsia Philippines'  (ORG)
  'January 2016'  (TIME)


In [6]:
# labels: gold relations, referencing entities by their vertexSet index (head/tail),
# with relation_id being a real Wikidata property ID (e.g. "P17" = country)
gold_relations = list(zip(
    doc["labels"]["head"], doc["labels"]["tail"],
    doc["labels"]["relation_id"], doc["labels"]["relation_text"],
))

print(f"{len(gold_relations)} gold relations:")
for h, t, rid, rtext in gold_relations:
    head_name = doc["vertexSet"][h][0]["name"]
    tail_name = doc["vertexSet"][t][0]["name"]
    print(f"  ({head_name!r}) --[{rid}: {rtext}]--> ({tail_name!r})")

13 gold relations:
  ('Zest Airways, Inc.') --[P159: headquarters location]--> ('Pasay City')
  ('Zest Airways, Inc.') --[P17: country]--> ('Philippines')
  ('Zest Air') --[P17: country]--> ('Philippines')
  ('Pasay City') --[P17: country]--> ('Philippines')
  ('Pasay City') --[P131: located in the administrative territorial entity]--> ('Metro Manila')
  ('Philippines') --[P150: contains administrative territorial entity]--> ('Metro Manila')
  ('Manila') --[P17: country]--> ('Philippines')
  ('Metro Manila') --[P150: contains administrative territorial entity]--> ('Pasay City')
  ('Metro Manila') --[P131: located in the administrative territorial entity]--> ('Philippines')
  ('Metro Manila') --[P17: country]--> ('Philippines')
  ('Ninoy Aquino International Airport') --[P131: located in the administrative territorial entity]--> ('Pasay City')
  ('Ninoy Aquino International Airport') --[P17: country]--> ('Philippines')
  ('Asian Spirit') --[P17: country]--> ('Philippines')


### Step 2: Run a baseline extractor (spaCy NER) on the raw document text

In [7]:
nlp = spacy.load("en_core_web_sm")
spacy_doc = nlp(doc_text)

baseline_entities = [(ent.text, ent.label_) for ent in spacy_doc.ents]
print(f"{len(baseline_entities)} spaCy entities:")
for name, etype in baseline_entities:
    print(f"  {name!r}  ({etype})")

28 spaCy entities:
  'Zest Airways , Inc.'  (ORG)
  'AirAsia Zest'  (ORG)
  'Asian'  (NORP)
  'Zest Air'  (PERSON)
  'the Ninoy Aquino International Airport'  (FAC)
  'Pasay City'  (GPE)
  'Philippines'  (GPE)
  'Manila'  (GPE)
  'Cebu'  (GPE)
  '24'  (CARDINAL)
  '2013'  (DATE)
  'Philippines AirAsia'  (ORG)
  'Ninoy Aquino International Airport'  (PERSON)
  'Manila'  (GPE)
  'Asian'  (NORP)
  'first'  (ORDINAL)
  'Philippines'  (GPE)
  'August 16 , 2013'  (DATE)
  'the Civil Aviation Authority of the Philippines'  (ORG)
  'CAAP'  (ORG)
  'the Government of the Republic of the Philippines'  (ORG)
  'Zest Air'  (ORG)
  'Less than a year'  (DATE)
  'AirAsia'  (GPE)
  "Zest Air 's"  (ORG)
  'AirAsia Zest'  (ORG)
  'AirAsia Philippines'  (LOC)
  'January 2016'  (DATE)


### Step 3: Compare baseline extraction against DocRED's gold, Wikidata-mapped entities

In [8]:
import re

def normalize_entity(name):
    """Undo tokenization/span artifacts that cause exact-match comparison to
    understate real overlap: space-before-punctuation from `" ".join(tokens)`,
    leading articles, and trailing possessives."""
    name = name.lower().strip()
    name = re.sub(r"\s+([,.;:'])", r"\1", name)   # "zest air , inc." -> "zest air, inc."
    name = re.sub(r"^(the|a|an)\s+", "", name)     # "the ninoy aquino ..." -> "ninoy aquino ..."
    name = re.sub(r"'s$", "", name)                # "zest air's" -> "zest air"
    return name.strip()

gold_names = {normalize_entity(name) for name, _ in gold_entities}
baseline_names = {normalize_entity(name) for name, _ in baseline_entities}

matched = gold_names & baseline_names
missed_by_baseline = gold_names - baseline_names       # gold entities spaCy didn't find
not_in_schema = baseline_names - gold_names            # spaCy entities with no gold/schema counterpart

print(f"Gold entities matched by spaCy: {len(matched)}/{len(gold_names)}")
print(f"\nGold entities spaCy MISSED:")
for n in missed_by_baseline:
    print(f"  {n!r}")
print(f"\nspaCy entities with no gold/schema match (candidate coverage gaps or noise):")
for n in not_in_schema:
    print(f"  {n!r}")

Gold entities matched by spaCy: 14/17

Gold entities spaCy MISSED:
  'metro manila'
  'year'
  'asian spirit'

spaCy entities with no gold/schema match (candidate coverage gaps or noise):
  'august 16, 2013'
  'caap'
  'first'
  'asian'
  'airasia zest'
  'less than a year'
  'government of the republic of the philippines'


### Step 4: Findings write-up

**Setup**
- Dataset: [DocRED](https://huggingface.co/datasets/thunlp/docred), loaded from the `refs/convert/parquet` revision (the dataset's legacy loading script is no longer supported by current `datasets` versions). The converted `train` split merges the original `train_annotated` (3,053 gold docs) and `train_distant` (101,873 weakly-supervised docs) into one split — explicitly sliced to the first 3,053 rows to guarantee gold-quality annotations.
- Baseline extractor: spaCy `en_core_web_sm` NER, run on the raw document text (ignoring DocRED's gold labels).
- Probe document: "AirAsia Zest" — 17 gold entities, 13 gold relations (Wikidata property IDs, e.g. `P17` = country, `P131` = located in the administrative territorial entity).

**Results**
- spaCy correctly matched **14/17 (82%)** gold entities once the comparison was normalized to strip tokenization artifacts (space-before-punctuation from the token-joined text, leading articles, trailing possessives) — the raw exact-string match had understated this at 12/17.
- 3 genuine misses: `metro manila` (present in the text, simply untagged by spaCy), `asian spirit` (spaCy tagged only the `Asian` fragment as NORP, never joined it into the full org name), and `a year` vs. spaCy's wider span `less than a year` (a boundary mismatch our normalization doesn't cover).
- spaCy also surfaced entities with no counterpart in our gold-name list (`caap`, `airasia zest`, `government of the republic of the philippines`, plus date/ordinal noise like `first`, `august 16, 2013`).

**Caveat: the gold-entity list itself is incomplete.** `gold_entities` was built from only the *first* mention name per DocRED entity (`vertexSet[i][0]`), but each gold entity typically has multiple aliases across its mentions in the document. `caap` and `airasia zest` are very likely legitimate aliases of entities already in the gold set (Civil Aviation Authority of the Philippines, and the document's own subject) rather than spaCy false positives — but this run can't tell the difference, because the alias mentions were discarded when building the comparison set. This means the "no gold match" list currently conflates real coverage gaps with an artifact of our own extraction code, and the true false-positive rate is likely lower than it appears.

**Relation extraction was not attempted in this pass** — only entity extraction was benchmarked. spaCy has no built-in relation extraction; testing that would require either a separate relation-extraction model/prompt-based LLM approach, or dependency-parse heuristics.

**Open next steps**
1. Rebuild `gold_entities` from *all* mention names per entity (not just the first) and re-run the comparison — expected to raise the effective match rate and shrink the "no gold match" list.
2. Extend past entity extraction into relation extraction (e.g. prompt an LLM to output `(head, relation, tail)` triples, or use a dependency-parse heuristic) and compare against the 13 gold Wikidata relations.
3. Scale from a single probe document to a small sample (e.g. 20–30 docs from `train_annotated`) to get an aggregate precision/recall number instead of one anecdotal case — this feeds directly into Task 5 (benchmarking/metrics).
4. Once schema coverage gaps are more reliably identified (post alias-fix), log them explicitly as candidate additions to the project's own ontology, rather than treating every non-match as noise.

**Conclusion:** the prototype pipeline works end-to-end — dataset load, gold inspection, baseline extraction, and comparison against a real Wikidata-based schema — and spaCy's out-of-the-box NER already recovers a solid majority of gold entities on this sample. The main limitation is in the *evaluation harness* itself (single-mention gold names, no relation extraction, single-document sample), not yet in the extraction quality — those are the priorities before this can be trusted as a real benchmark.